In [1]:
from Bprice import *

Initializing Bprice Package Success


In [2]:

import pandas as pd
import csv


In [ ]:
from binance.client import Client
from getpass import getpass
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
from datetime import datetime
from binance.enums import SIDE_BUY, SIDE_SELL, ORDER_TYPE_MARKET,ORDER_TYPE_LIMIT
import joblib
from Bprice import *
from Bprice.advanced_ta import *

In [3]:
# from ta.trend import ema_indicator as EMA, sma_indicator as SMA

df = pd.read_csv('btc_price.csv')

In [7]:
df

,Unnamed: 0,open,high,low,close,volume,datetime
0,0,63074.00,63094.01,62742.00,62840.73,768.12,2024-05-10 00:00:00
1,1,62840.72,62968.00,62662.07,62883.50,776.17,2024-05-10 01:00:00
2,2,62883.50,63043.29,62771.76,62910.61,605.45,2024-05-10 02:00:00
3,3,62910.62,62932.15,62793.94,62916.16,586.05,2024-05-10 03:00:00
4,4,62916.16,63008.00,62793.04,62929.99,607.57,2024-05-10 04:00:00
...,...,...,...,...,...,...,...
5025,5025,102764.01,103159.76,102565.63,102698.24,2204.40,2024-12-05 09:00:00
5026,5026,102698.25,102920.00,102260.00,102280.62,1591.37,2024-12-05 10:00:00
5027,5027,102280.62,102672.84,102237.79,102566.91,1395.71,2024-12-05 11:00:00
5028,5028,102566.91,103295.99,102452.95,102846.15,2215.64,2024-12-05 12:00:00


In [8]:
from Bprice.advanced_ta import LorentzianClassification

# from ta.volume import money_flow_index as MFI
lc = LorentzianClassification(
        df,
        
        settings=LorentzianClassification.Settings(
            source=df['close'],
            neighborsCount=21,
            maxBarsBack=4000,
            useDynamicExits=False
        ),
        
        filterSettings=LorentzianClassification.FilterSettings(
            useVolatilityFilter=True,
            useRegimeFilter=True,
            useAdxFilter=True,
            regimeThreshold=-0.1,
            adxThreshold=20,
            
            kernelFilter = LorentzianClassification.KernelFilter(
                useKernelSmoothing = False,
                lookbackWindow = 21,
                relativeWeight = 14.0,
                regressionLevel = 50,
                crossoverLag = 2
            )
        ))

true


ValueError: Data length must be greater than or equal to the period.

In [ ]:
def Get_sma(data, period):
    if not isinstance(data, (list, tuple, pd.Series)):
        data = list(data)
    if len(data) < period:
        raise ValueError("Data length must be greater than or equal to the period.")
    
    # Calculate SMA
    sma = [np.nan] * (period - 1)  # Fill with NaN for initial periods
    sma.extend([sum(data[i:i + period]) / period for i in range(len(data) - period + 1)])
    
    # Return as DataFrame
    df = pd.DataFrame({'sma'+str(period): sma})
    return df

In [ ]:


def Get_rsi(data, period=14):
    if not isinstance(data, (list, tuple, pd.Series)):
        data = list(data)
    if len(data) < period + 1:
        raise ValueError("Data length must be greater than the specified period.")
    
    # Convert to Pandas Series for easier calculation
    data = pd.Series(data)
    
    # Calculate price changes
    delta = data.diff()
    
    # Separate gains and losses
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    # Calculate average gain and loss using rolling mean
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    
    # Calculate the Relative Strength (RS)
    rs = avg_gain / avg_loss
    
    # Calculate the RSI
    rsi = 100 - (100 / (1 + rs))
    
    # Return as a DataFrame
    df = pd.DataFrame({'RSI': rsi})
    return df


In [ ]:
def Get_cci(high, low, close, period=20):
    # Ensure inputs are valid
    if not all(isinstance(arr, (list, tuple, pd.Series)) for arr in [high, low, close]):
        raise ValueError("High, low, and close prices must be lists, tuples, or Pandas Series.")
    if len(high) != len(low) or len(low) != len(close):
        raise ValueError("High, low, and close prices must have the same length.")
    if len(high) < period:
        raise ValueError("Data length must be greater than or equal to the specified period.")
    
    # Convert to Pandas Series for easier calculation
    high = pd.Series(high)
    low = pd.Series(low)
    close = pd.Series(close)
    
    # Calculate the Typical Price
    typical_price = (high + low + close) / 3
    
    # Calculate the Simple Moving Average (SMA) of the Typical Price
    sma = typical_price.rolling(window=period).mean()
    
    # Calculate the Mean Deviation
    mean_deviation = typical_price.rolling(window=period).apply(
        lambda x: np.mean(np.abs(x - np.mean(x))), raw=True
    )
    
    # Calculate the CCI
    cci = (typical_price - sma) / (0.015 * mean_deviation)
    
    # Return as a DataFrame
    df = pd.DataFrame({'CCI': cci})
    return df

In [ ]:


def calculate_adx(high, low, close, window=14,get_di = False,fillna=False):
    # Calculate True Range (TR)
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # Calculate +DM and -DM
    plus_dm = high.diff().clip(lower=0)
    minus_dm = low.diff().clip(upper=0).abs()

    # Smooth TR, +DM, and -DM using Wilder's method
    atr = tr.ewm(alpha=1/window, adjust=False).mean()
    plus_dm_smoothed = plus_dm.ewm(alpha=1/window, adjust=False).mean()
    minus_dm_smoothed = minus_dm.ewm(alpha=1/window, adjust=False).mean()

    # Calculate +DI and -DI
    plus_di = 100 * (plus_dm_smoothed / atr)
    minus_di = 100 * (minus_dm_smoothed / atr)

    # Calculate DX
    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)

    # Calculate ADX
    adx = dx.ewm(alpha=1/window, adjust=False).mean()

    # Handle NaN values if needed
    if fillna:
        adx = adx.fillna(20)
        plus_di = plus_di.fillna(20)
        minus_di = minus_di.fillna(20)
    if get_di:
        
        return adx, plus_di, minus_di
    else:
        return adx


In [ ]:

def calculate_atr(high: pd.Series, low: pd.Series, close: pd.Series, window: int = 14, fillna: bool = False) -> pd.Series:

    # Calculate True Range (TR)
    close_shift = close.shift(1)
    tr1 = high - low
    tr2 = abs(high - close_shift)
    tr3 = abs(low - close_shift)
    true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    
    # Calculate ATR using Wilder's smoothing method
    atr = true_range.ewm(alpha=1/window, adjust=False).mean()

    # Handle NaN values if needed
    if fillna:
        atr = atr.fillna(0)

    return pd.Series(atr, name="atr")


In [ ]:

from ta.volatility import average_true_range as ATR
from ta.trend import cci as CCI, adx as ADX

In [ ]:
ATR(df['high'],df['low'],df['close'],14)

In [ ]:
calculate_atr(df['high'],df['low'],df['close'],14)

In [ ]:
ADX(df['high'],df['low'],df['close'],14)

In [ ]:
adx

In [ ]:
adx

In [ ]:
adx = calculate_adx(df['high'],df['low'],df['close'], window=14, fillna=False)

In [ ]:
df['adx'] = calculate_adx(df['high'],df['low'],df['close'],14)

In [ ]:
df

In [ ]:
df['close']

In [ ]:
x= Get_Rsi(25, df['close'])

In [ ]:
ema_df = Get_ema(df['close'], 25)

In [ ]:
ema_df

In [ ]:
talib = EMA(df['close'],25)

In [ ]:
ema_df['test'] = talib

In [ ]:
ema_df

In [ ]:
df = pd.DataFrame(mylib, columns=["Values"])

In [ ]:
df['1'] = talib

In [ ]:
mylib = Get_ema(df['close'],25)

In [ ]:
mylib